In [ ]:
import pandas as pd
from pathlib import Path
import sys

"""
Script to process logs by session.
It performs the following steps:
1. Iterates over each Subject folder (e.g., 'S01', 'S02').
2. Inside each Subject, iterates over each "session" subfolder
   (the "grandmother", e.g., 'test1a', 'test1b').
3. For EACH session:
   a. Finds all 'predictions_log.csv' files inside it
      (at any depth).
   b. Extracts 'class' and 'timestamp' columns.
   c. Concatenates all data for that session.
   d. Sorts the combined DataFrame by 'timestamp'.
   e. Saves the file in '.../data/Ssubject/steady_state/'
      named 'predictions_log_sessionname.csv'.
"""

def process_by_session(base_repo_path, output_base_dir):
    
    # 1. Path Definitions
    
    # INPUT Path: Where data is located
    base_path = Path(base_repo_path)
    
    # OUTPUT Path: Absolute base path specified by user
    output_base_path = Path(output_base_dir)
    
    # Columns to extract
    required_columns = ['class', 'timestamp']

    print(f"Starting data search in: {base_path}")
    print(f"Output base path: {output_base_path}\n")

    # 2. Find all Subject folders
    s_folders = sorted([f for f in base_path.iterdir() if f.is_dir() and f.name.startswith('S')])
    
    if not s_folders:
        print("Warning: No folders starting with 'S' found.")
        return

    print(f"Found {len(s_folders)} subject folders. Starting processing...")

    # 3. Iterate over each Subject (e.g. S03)
    for subject_folder in s_folders:
        subject_name = subject_folder.name
        print(f"\n--- Processing Subject: {subject_name} ---")
        
        # Define base output folder for this subject
        subject_output_dir = output_base_path / subject_name / "steady_state"
        
        # Create output folder if it doesn't exist
        try:
            subject_output_dir.mkdir(parents=True, exist_ok=True)
        except Exception as e:
            print(f"  > ERROR: Cannot create output folder {subject_output_dir}")
            print(f"    Details: {e}")
            continue # Skip to next subject
            
        # 4. Find all "session" folders (grandmothers) inside the subject
        session_folders = [f for f in subject_folder.iterdir() if f.is_dir()]
        
        if not session_folders:
            print(f"  > Warning: No 'session' folders found for {subject_name}.")
            continue
            
        print(f"  > Found {len(session_folders)} sessions: {[f.name for f in session_folders]}")

        # 5. Iterate over each Session (e.g. test1a)
        for session_folder in session_folders:
            session_name = session_folder.name
            print(f"    > Processing session: {session_name}")
            
            # List for data of THIS session
            session_data_list = []
            
            # Output filename based on session
            output_csv_file = subject_output_dir / f"predictions_log_{session_name}.csv"
            
            # 6. Find all CSVs inside the session folder (recursively)
            csv_files_in_session = list(session_folder.rglob("predictions_log.csv"))
            
            if not csv_files_in_session:
                print(f"      > No 'predictions_log.csv' found in {session_name}.")
                continue # Go to next session

            files_found = 0
            for csv_file in csv_files_in_session:
                try:
                    df = pd.read_csv(
                        csv_file, 
                        usecols=required_columns, 
                        skipinitialspace=True,
                        dtype={'timestamp': str}
                    )
                    
                    if not df.empty:
                        session_data_list.append(df)
                        files_found += 1
                    
                except pd.errors.EmptyDataError:
                    print(f"      > Warning: Empty file ignored {csv_file.relative_to(session_folder)}")
                except ValueError as e:
                    print(f"      > ERROR: Columns not found in {csv_file.relative_to(session_folder)}.")
                except Exception as e:
                    print(f"      > Error reading {csv_file.relative_to(session_folder)}: {e}")
            
            if not session_data_list:
                print(f"      > No valid data read for session {session_name}.")
                continue
                
            print(f"      > Read {files_found} files. Concatenating and sorting...")

            # 7. Concatenate, Sort, and Save for THIS session
            try:
                combined_df = pd.concat(session_data_list, ignore_index=True)
                combined_df_sorted = combined_df.sort_values(by='timestamp')
                
                # Save CSV
                combined_df_sorted.to_csv(output_csv_file, index=False)
                print(f"      > File saved successfully for {session_name} to:\n        {output_csv_file}")
                
            except Exception as e:
                print(f"      > ERROR saving file for {session_name}: {e}")

    print("\n--- Session processing complete ---")

# --- SCRIPT START ---
if __name__ == "__main__":
    
    # INPUT Path: where data to analyze is located.
    base_data_path = r"C:\Users\nicol\Thesis\DATA real time\_test_raw"
    
    # OUTPUT Path: absolute base folder to save results
    base_output_path = r"C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data"
    
    process_by_session(base_data_path, base_output_path)

Inizio ricerca dati in: C:\Users\nicol\Thesis\DATA real time\_test_raw
Percorso base di output: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data

Trovate 12 cartelle soggetto. Inizio elaborazione...

--- Processando Soggetto: S03 ---
  > Trovate 5 sessioni: ['calib', 'test1a', 'test1b', 'test2a', 'test2b']
    > Processando sessione: calib
      > Nessun 'predictions_log.csv' trovato in calib.
    > Processando sessione: test1a
      > Letti 1 file. Ora concateno e ordino...
      > File salvato con successo per test1a in:
        C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S03\steady_state\predictions_log_test1a.csv
    > Processando sessione: test1b
      > Letti 1 file. Ora concateno e ordino...
      > File salvato con successo per test1b in:
        C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S03\steady_state\predictions_log_test1b.csv
    > Processando sessione: test2a
      > Letti 1 file. Ora concateno e ordino...
      > File salvato con successo per test2a in:
  

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix
import sys

"""
Script to evaluate subject performance (metrics batch).
It performs the following steps:
1. Iterates over all Subject folders (S*) in the base path.
2. For each subject, looks for 'predictions_log_*.csv' files in 'steady_state/'.
3. For each session log:
   a. Loads the data and filters valid rows (requires 'class' and 'class_real').
   b. Calculates the Confusion Matrix (absolute values).
   c. Saves the Confusion Matrix to 'confusion_matrix_absolute.csv'.
   d. Generates and saves a normalized Confusion Matrix plot (.png)
      with specific font sizes for readability.
"""

def plot_confusion_matrix(cm_abs, class_names, title, output_path):
    """
    Helper function to create and save a confusion matrix plot.
    Shows percentages (row-normalized) with large fonts.
    """
    try:
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Calculate row-normalized percentages (True Label)
        cm_sum = cm_abs.sum(axis=1)[:, np.newaxis]
        
        # Avoid division by zero if a class has no real samples
        cm_perc = np.nan_to_num(cm_abs.astype('float') / cm_sum)
        
        # Formatting for annotations (e.g. "95.20%")
        plot_fmt = '.2%' 
        
        # Font settings
        annot_font_size = 22
        title_font_size = 26
        label_font_size = 24
        tick_font_size = 20
        
        sns.heatmap(
            cm_perc, 
            annot=True, 
            fmt=plot_fmt, 
            cmap='Blues', 
            xticklabels=class_names, 
            yticklabels=class_names,
            ax=ax,
            annot_kws={"size": annot_font_size} # Set cell font size
        )
        
        ax.set_xlabel('Predicted Label', fontsize=label_font_size)
        ax.set_ylabel('True Label', fontsize=label_font_size)
        ax.set_title(title, fontsize=title_font_size)
        
        ax.tick_params(axis='x', labelsize=tick_font_size)
        ax.tick_params(axis='y', labelsize=tick_font_size, rotation=0)
        
        plt.tight_layout()
        plt.savefig(output_path)
        plt.close(fig) 
        
        print(f"  > Confusion Matrix Plot (Percentage) saved to:\n    {output_path}")

    except Exception as e:
        print(f"  > ERROR creating plot: {e}")

def evaluate_subject_performance(subject_id, base_data_dir):
    """
    Generates an evaluation report for a single subject.
    Finds 'predictions_log_*.csv' files, calculates metrics,
    and saves results in dedicated folders.
    """
    
    # 1. Path Definitions
    input_dir = Path(base_data_dir) / subject_id / "steady_state"
    output_eval_base_dir = input_dir / "evaluation"
    
    CLASS_NAMES = ['no weight', 'light', 'medium', 'heavy']
    CLASS_LABELS = [0, 1, 2, 3]

    print(f"--- Starting Evaluation for Subject: {subject_id} ---")
    print(f"Input folder: {input_dir}")
    print(f"Output base folder: {output_eval_base_dir}\n")

    # 2. Find all session log files
    log_files = list(input_dir.glob("predictions_log_*.csv"))
    
    if not log_files:
        print(f"Warning: No 'predictions_log_*.csv' files found in {input_dir}")
        return

    print(f"Found {len(log_files)} log files to analyze...")

    # 3. Iterate over each log file
    for file_path in log_files:
        
        # Extract session name (e.g., test1a from predictions_log_test1a.csv)
        session_name = file_path.stem.replace("predictions_log_", "")
        print(f"\n--- Processing session: {session_name} ---")

        # 4. Create dedicated output folder for this session
        session_output_dir = output_eval_base_dir / session_name
        try:
            session_output_dir.mkdir(parents=True, exist_ok=True)
        except Exception as e:
            print(f"  > ERROR: Cannot create output folder {session_output_dir}")
            print(f"    Details: {e}")
            continue 

        # 5. Read Data
        try:
            df = pd.read_csv(file_path)
            
            # Check for required columns
            if 'class' not in df.columns or 'class_real' not in df.columns:
                print(f"  > ERROR: File {file_path.name} is missing 'class' or 'class_real'. Skipping.")
                continue
                
            df.dropna(subset=['class_real'], inplace=True)
            
            y_true = df['class_real'].astype(int)
            y_pred = df['class'].astype(int)
            
        except Exception as e:
            print(f"  > ERROR reading file {file_path.name}: {e}")
            continue

        if y_true.empty:
            print(f"  > Warning: No valid data (no NaN) found in {file_path.name}.")
            continue
            
        # 6. Calculate Metrics (Confusion Matrix Only)
        print(f"  > Calculating confusion matrix for {len(y_true)} samples...")
        
        # Confusion Matrix (Absolute values)
        cm_abs = confusion_matrix(
            y_true, y_pred, 
            labels=CLASS_LABELS
        )

        # 8. Save Confusion Matrix to CSV
        cm_csv_path = session_output_dir / "confusion_matrix_absolute.csv"
        try:
            cm_df = pd.DataFrame(
                cm_abs, 
                index=[f"True_{name}" for name in CLASS_NAMES], 
                columns=[f"Pred_{name}" for name in CLASS_NAMES]
            )
            cm_df.to_csv(cm_csv_path)
            print(f"  > Confusion Matrix (Absolute CSV) saved to: {cm_csv_path.name}")
        except Exception as e:
            print(f"  > ERROR saving Matrix CSV: {e}")

        # 9. Create and Save Plot
        plot_png_path = session_output_dir / "confusion_matrix_plot.png"
        
        plot_title = f"Confusion Matrix - {session_name}\nSubject {subject_id} (Row-Normalized %)"
        
        plot_confusion_matrix(
            cm_abs, # Pass absolute values
            CLASS_NAMES, 
            plot_title, 
            plot_png_path
        )

    # print(f"\n--- Evaluation complete for {subject_id} ---") 

# --- SCRIPT START ---
if __name__ == "__main__":
    
    # Base path where folders S01, S02, S03... are located
    PERCORSO_BASE_DATA = r"C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data"
    base_data_path = Path(PERCORSO_BASE_DATA)

    if not base_data_path.is_dir():
        print(f"ERROR: Base path does not exist: {PERCORSO_BASE_DATA}")
    else:
        print(f"Starting batch processing in folder: {base_data_path}\n")
        
        # Search for all folders starting with 'S' (e.g., S01, S02, ...)
        # and ensure they are directories. Sort them.
        subjects_found = sorted(
            [d for d in base_data_path.glob("S*") if d.is_dir()]
        )
        
        if not subjects_found:
            print(f"No subject folders (S*) found in {base_data_path}")
        else:
            print(f"Found {len(subjects_found)} subjects: {[s.name for s in subjects_found]}\n")
            
            # Iterate over each subject and start evaluation
            for subject_path in subjects_found:
                subject_id = subject_path.name
                evaluate_subject_performance(subject_id, PERCORSO_BASE_DATA)
                print(f"--- Finished evaluation for {subject_id} ---\n")
        
        print("--- Batch processing complete ---")

Avvio elaborazione batch nella cartella: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data

Trovati 12 soggetti: ['S03', 'S08', 'S10', 'S12', 'S13', 'S15', 'S16', 'S17', 'S18', 'S19', 'S20', 'S21']

--- Inizio Valutazione per Soggetto: S03 ---
Cartella di input: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S03\steady_state
Cartella base di output: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S03\steady_state\evaluation

Trovati 4 file di log da analizzare...

--- Processando sessione: test1a ---
  > Calcolo matrice di confusione per 3196 campioni...
  > Matrice di confusione (CSV Assoluta) salvata in: confusion_matrix_absolute.csv
  > Plot matrice di confusione (PERCENTUALE) salvato in:
    C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\S03\steady_state\evaluation\test1a\confusion_matrix_plot.png

--- Processando sessione: test1b ---
  > Calcolo matrice di confusione per 2769 campioni...
  > Matrice di confusione (CSV Assoluta) salvata in: confusion_matrix_absolute.csv
  >

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import random
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

"""
Script to evaluate subject performance (metrics batch) with UNDERSAMPLING.
It performs the following steps:
1. Iterates over all Subject folders (S*) in the base path.
2. Aggregates confusion matrices from all sessions (test1a, test1b, etc.).
3. Performs analysis on:
   - Individual Sessions
   - Combined Test 1 (1a + 1b)
   - Combined Test 2 (2a + 2b)
   - Overall Total
4. For each analysis set:
   - Saves Unbalanced CSV.
   - Performs UNDERSAMPLING (balancing classes based on minority class).
   - Saves Undersampled CSV.
   - Generates Seaborn Heatmap (without title, normalized %).
   - Calculates 4-class and 3-class metrics.
5. Saves a single aggregated metrics report.
"""

# --- GLOBAL CONSTANTS ---
CLASS_NAMES_FULL = ['no weight', 'light', 'medium', 'heavy']
CLASS_LABELS_FULL = [0, 1, 2, 3] 
TEST_SESSIONS = ['test1a', 'test1b', 'test2a', 'test2b']

# Constants for 3-class analysis
CLASS_NAMES_3 = ['light', 'medium', 'heavy']
CLASS_LABELS_3 = [1, 2, 3]

# Plotting constants
DPI = 300
FIGSIZE = (8, 6) 

# ==================
# --- HELPER FUNCTIONS (CSV, METRICS, UNDERSAMPLING) ---
# ==================

def save_cm_csv_only(cm_abs, output_dir, file_name_prefix, class_names):
    """Saves a confusion matrix (absolute) to CSV only."""
    cm_csv_path = output_dir / f"{file_name_prefix}_absolute.csv"
    try:
        cm_df = pd.DataFrame(
            cm_abs, 
            index=[f"True_{name}" for name in class_names], 
            columns=[f"Pred_{name}" for name in class_names]
        )
        cm_df.to_csv(cm_csv_path)
        print(f"    > CSV saved: {cm_csv_path.name}")
    except Exception as e:
        print(f"    > ERROR saving CSV: {e}")

def reconstruct_predictions(cm):
    """
    Reconstructs y_true and y_pred lists from a confusion matrix.
    Returns shuffled lists.
    """
    pairs = []
    for true_idx, row in enumerate(cm):
        for pred_idx, count in enumerate(row):
            # Add 'count' pairs of (true_idx, pred_idx)
            pairs.extend([(true_idx, pred_idx)] * int(count))
    
    # Shuffle the pairs
    random.shuffle(pairs)
    
    y_true = [p[0] for p in pairs]
    y_pred = [p[1] for p in pairs]
    return y_true, y_pred

def perform_undersampling(y_true, y_pred, class_labels):
    """
    Performs undersampling based on the minority class (y_true).
    """
    data = list(zip(y_true, y_pred))
    
    # Calculate counts for each REAL class
    true_counts = np.bincount(y_true, minlength=len(class_labels))
    
    # Find the minority class size (excluding classes with 0 samples)
    valid_counts = true_counts[true_counts > 0]
    if len(valid_counts) == 0:
        return [], []

    min_samples = np.min(valid_counts)
    
    print(f"    > Performing undersampling: {min_samples} samples per class.")
    
    undersampled_data = []
    # Dictionary to track how many samples we have for each class
    class_counters = {label: 0 for label in class_labels}
    
    # Iterate over shuffled data
    for yt, yp in data:
        # If we haven't reached the limit for this true class
        if class_counters[yt] < min_samples:
            undersampled_data.append((yt, yp))
            class_counters[yt] += 1
            
    if not undersampled_data:
        return [], []
        
    y_true_us = [p[0] for p in undersampled_data]
    y_pred_us = [p[1] for p in undersampled_data]
    
    return y_true_us, y_pred_us

def calculate_and_format_metrics(y_true_us, y_pred_us, title, class_labels_4, class_names_4, class_labels_3, class_names_3):
    """
    Calculates and formats metrics for 4-class and 3-class analysis.
    """
    if not y_true_us:
        return f"--- {title} ---\nNo data to analyze.\n\n"

    buffer = []
    buffer.append("=" * 80)
    buffer.append(f"--- METRICS FOR: {title} ---")
    buffer.append("=" * 80)
    buffer.append(f"(Based on {len(y_true_us)} total samples after undersampling)\n")

    # --- 1. 4-Class Metrics ---
    buffer.append("\n--- 4-Class Analysis (no weight, light, medium, heavy) ---")
    
    acc_4_class = accuracy_score(y_true_us, y_pred_us)
    report_4_class = classification_report(
        y_true_us, y_pred_us,
        labels=class_labels_4,
        target_names=class_names_4,
        zero_division=0
    )
    buffer.append(f"\nOverall Accuracy (4-Class): {acc_4_class:.4f}")
    buffer.append("\nClassification Report (4-Class):\n")
    buffer.append(report_4_class)

    # --- 2. 3-Class Metrics (light, medium, heavy) ---
    buffer.append("\n\n--- 3-Class Analysis (light, medium, heavy) ---")
    
    # Filter the lists to calculate 3-class accuracy
    y_true_3class = []
    y_pred_3class = []
    for yt, yp in zip(y_true_us, y_pred_us):
        # Consider only samples that *should* be 1, 2, or 3
        if yt in class_labels_3:
            y_true_3class.append(yt)
            y_pred_3class.append(yp)

    if y_true_3class:
        acc_3_class = accuracy_score(y_true_3class, y_pred_3class)
        buffer.append(f"\nOverall Accuracy (Classes 1, 2, 3 only): {acc_3_class:.4f}")
    else:
        buffer.append("\nOverall Accuracy (Classes 1, 2, 3 only): N/A (no samples)")

    # The classification report handles labels automatically
    report_3_class = classification_report(
        y_true_us, y_pred_us,
        labels=class_labels_3,
        target_names=class_names_3,
        zero_division=0
    )
    buffer.append("\nClassification Report (Classes 1, 2, 3):\n")
    buffer.append(report_3_class)
    buffer.append("\n" * 2)

    return "\n".join(buffer)

# ==================
# --- PLOTTING FUNCTION (SEABORN STYLE) ---
# ==================

def save_seaborn_confusion_matrix(
    cm_abs, 
    class_names, 
    filename, 
    figsize=FIGSIZE, 
    dpi=DPI,
    label_size=22,
    tick_size=20,
    annot_size=20,
    cbar_tick_size=20
):
    """
    Generates and saves a Seaborn-style confusion matrix
    with customizable fonts, row-normalized (0-100), 
    and WITHOUT TITLE.
    """
    try:
        # Calculate row percentages
        row_sums = cm_abs.sum(axis=1, keepdims=True)
        # Avoid division by 0
        with np.errstate(divide='ignore', invalid='ignore'):
            cm_percent = np.nan_to_num(cm_abs.astype('float') / row_sums) * 100
        
        # Create figure
        plt.figure(figsize=figsize)
        
        # 1. Create heatmap
        ax = sns.heatmap(
            cm_percent, 
            annot=True, 
            fmt='.2f', 
            cmap='Blues', 
            xticklabels=class_names, 
            yticklabels=class_names,
            annot_kws={'size': annot_size}, 
            vmin=0,                          
            vmax=100                         
        )
        
        # 2. Set labels (WITHOUT TITLE)
        ax.set_xlabel('Predicted', fontsize=label_size) 
        ax.set_ylabel('True', fontsize=label_size)        

        # 3. Set ticks
        ax.tick_params(axis='x', labelsize=tick_size) 
        ax.tick_params(axis='y', labelsize=tick_size) 
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

        # 4. Modify colorbar
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=cbar_tick_size) 

        # 5. Save and close
        plt.tight_layout() 
        plt.savefig(filename, dpi=dpi, bbox_inches='tight')
        plt.close() 
        print(f"    > Seaborn plot saved: {filename.name}")

    except Exception as e:
        print(f"    > ERROR plotting with Seaborn: {e}")


# ==================
# --- MAIN SCRIPT ---
# ==================
def main():
    # Base path setup
    BASE_DATA_PATH = r"C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data"
    base_data_path = Path(BASE_DATA_PATH)
    
    # Root output directory for all aggregated steady state analysis
    output_dir_base = base_data_path / "all" / "steady_state" 
    output_dir_base.mkdir(parents=True, exist_ok=True)
    
    print(f"--- Starting Full Aggregation, Metrics, and Plotting ---")
    print(f"Base input directory: {base_data_path}")
    print(f"Base output directory: {output_dir_base}\n")

    # Dictionary to accumulate confusion matrices
    session_matrices = {
        session: np.zeros((len(CLASS_LABELS_FULL), len(CLASS_LABELS_FULL)), dtype=int)
        for session in TEST_SESSIONS
    }
    
    subjects_found = sorted(
        [d for d in base_data_path.glob("S*") if d.is_dir()]
    )
    
    if not subjects_found:
        print("ERROR: No subject folders (S*) found.")
        return
        
    print(f"Found {len(subjects_found)} subjects. Starting data collection...")

    # --- 1. Data Collection ---
    subjects_processed = 0
    for subject_path in subjects_found:
        print(f"\nProcessing Subject: {subject_path.name}")
        subject_has_data = False
        
        for session_name in TEST_SESSIONS:
            cm_path = subject_path / "steady_state" / "evaluation" / session_name / "confusion_matrix_absolute.csv"
            
            if cm_path.exists():
                try:
                    cm_df = pd.read_csv(cm_path, index_col=0)  
                    cm_data_np = cm_df.to_numpy()
                    
                    if cm_data_np.shape == (len(CLASS_LABELS_FULL), len(CLASS_LABELS_FULL)):
                        session_matrices[session_name] += cm_data_np
                        print(f"  > Found and added: {session_name}")
                        subject_has_data = True
                    else:
                        print(f"  > WARNING: {cm_path.name} has unexpected shape {cm_data_np.shape}. Skipping.")
                except Exception as e:
                    print(f"  > ERROR reading {cm_path.name}: {e}")
            else:
                print(f"  > Not found: {session_name}")
        
        if subject_has_data:
            subjects_processed += 1

    print(f"\n--- Data collection complete. {subjects_processed} subjects had data. ---")

    # --- 2. Defining Analysis Sets (Individual and Combined) ---
    print("\n--- Defining analysis sets and output directories ---")

    # Calculate combined matrices
    cm_test1 = session_matrices['test1a'] + session_matrices['test1b']
    cm_test2 = session_matrices['test2a'] + session_matrices['test2b']
    cm_overall = cm_test1 + cm_test2

    # List to hold analysis sets: (Title, Matrix_CM, Output_Dir, File_Prefix)
    analysis_sets = []
    
    # Add individual sessions
    for session_name in TEST_SESSIONS:
        out_dir = output_dir_base / session_name
        out_dir.mkdir(parents=True, exist_ok=True)
        analysis_sets.append(
            (f"Session: {session_name}", session_matrices[session_name], out_dir, f"cm_{session_name}")
        )
        print(f"  > Added analysis: {session_name} -> {out_dir.name}")

    # Add combined sessions
    out_dir_t1 = output_dir_base / "Test1_Combined"
    out_dir_t1.mkdir(parents=True, exist_ok=True)
    analysis_sets.append(
        ("Test 1 (1a+1b)", cm_test1, out_dir_t1, "cm_total_test1")
    )
    print(f"  > Added analysis: Test 1 (1a+1b) -> {out_dir_t1.name}")

    out_dir_t2 = output_dir_base / "Test2_Combined"
    out_dir_t2.mkdir(parents=True, exist_ok=True)
    analysis_sets.append(
        ("Test 2 (2a+2b)", cm_test2, out_dir_t2, "cm_total_test2")
    )
    print(f"  > Added analysis: Test 2 (2a+2b) -> {out_dir_t2.name}")

    out_dir_ovr = output_dir_base / "Overall_Combined"
    out_dir_ovr.mkdir(parents=True, exist_ok=True)
    analysis_sets.append(
        ("Overall (1+2)", cm_overall, out_dir_ovr, "cm_total_overall")
    )
    print(f"  > Added analysis: Overall (1+2) -> {out_dir_ovr.name}")


    # --- 3. & 4. Execution Loop (CSV, Undersampling, Plot, Metrics) ---
    print("\n--- Starting analysis loop (Unbalanced CSV, Undersampling, Metrics, Plot) ---")
    
    metrics_file_path = output_dir_base / "metrics_report_undersampled.txt"
    all_metrics_str = []
    
    for title, cm_data, output_dir, file_prefix in analysis_sets:
        print(f"\n  Analyzing {title}... (Output dir: {output_dir.name})")
        
        # --- A. Save Unbalanced CSV ---
        save_cm_csv_only(
            cm_data, output_dir, f"{file_prefix}_unbalanced", CLASS_NAMES_FULL
        )

        # --- B. Reconstruct and Undersample ---
        y_true, y_pred = reconstruct_predictions(cm_data)
        
        if y_true:
            y_true_us, y_pred_us = perform_undersampling(y_true, y_pred, CLASS_LABELS_FULL)
            
            # --- C. Save CSV and Plot (only if data exists after undersampling) ---
            if y_true_us:
                cm_us = confusion_matrix(y_true_us, y_pred_us, labels=CLASS_LABELS_FULL)
                
                # Save Undersampled CSV
                save_cm_csv_only(
                    cm_us, 
                    output_dir, 
                    f"{file_prefix}_undersampled", 
                    CLASS_NAMES_FULL
                )
                
                # Save Undersampled Plot (New Seaborn Style)
                save_seaborn_confusion_matrix(
                    cm_us, 
                    CLASS_NAMES_FULL,
                    output_dir / f"{file_prefix}_undersampled_seaborn.png"
                )
            
            # --- D. Calculate Metrics (on undersampled data) ---
            metrics_str = calculate_and_format_metrics(
                y_true_us, y_pred_us, f"{title} - UNDERSAMPLED",
                CLASS_LABELS_FULL, CLASS_NAMES_FULL,
                CLASS_LABELS_3, CLASS_NAMES_3
            )
            all_metrics_str.append(metrics_str)
        
        else:
            print("    > No data found for this set. Skipping metrics.")
            all_metrics_str.append(f"--- METRICS FOR: {title} ---\nNo data.\n\n")

    # --- 5. Save Single Metrics Report ---
    try:
        with open(metrics_file_path, 'w', encoding='utf-8') as f:
            f.write("AGGREGATED METRICS REPORT (UNDERSAMPLED)\n")
            f.write(f"Data Directory: {base_data_path}\n")
            f.write(f"Subjects Analyzed: {subjects_processed}\n")
            f.write("="*80 + "\n\n")
            f.write("\n".join(all_metrics_str))
        print(f"\n--- Undersampled metrics report saved to: {metrics_file_path} ---")
    except Exception as e:
        print(f"\n--- ERROR saving metrics report: {e} ---")

    print("\n--- Aggregation, metrics, and plotting complete ---")

if __name__ == "__main__":
    main()

--- Starting Full Aggregation, Metrics, and Plotting ---
Base input directory: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data
Base output directory: C:\Users\nicol\Thesis\pyl_est\offline\myo_cv\data\all\steady_state

Found 12 subjects. Starting data collection...

Processing Subject: S03
  > Found and added: test1a
  > Found and added: test1b
  > Found and added: test2a
  > Found and added: test2b

Processing Subject: S08
  > Found and added: test1a
  > Found and added: test1b
  > Found and added: test2a
  > Found and added: test2b

Processing Subject: S10
  > Found and added: test1a
  > Found and added: test1b
  > Found and added: test2a
  > Found and added: test2b

Processing Subject: S12
  > Found and added: test1a
  > Found and added: test1b
  > Found and added: test2a
  > Found and added: test2b

Processing Subject: S13
  > Found and added: test1a
  > Found and added: test1b
  > Found and added: test2a
  > Found and added: test2b

Processing Subject: S15
  > Found and added: te